# 🎬 Netflix Content Strategy Analysis

This notebook provides a detailed exploratory data analysis (EDA) of the Netflix titles dataset. We explore content trends, genre distributions, and production footprints to understand the strategic direction of the streaming giant's library.

In [9]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import ast
import os

# Set Plotly template for dark theme
pio.templates.default = "plotly_dark"

## 1. Data Loading & Cleaning

We load the `titles.csv` dataset and perform initial cleaning, focusing on parsing list-like columns (genres and production countries).

In [10]:
def parse_list_col(val):
    try:
        if isinstance(val, str) and val.startswith('['):
            return ast.literal_eval(val)
        return val
    except:
        return val

df = pd.read_csv('titles.csv')

# Rename for clarity
df = df.rename(columns={'production_countries': 'country', 'genres': 'listed_in'})

# Parse lists
for col in ['country', 'listed_in']:
    if col in df.columns:
        df[col] = df[col].apply(parse_list_col)

# Extract year added if available, otherwise use release_year
if 'date_added' in df.columns:
    df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
    df['year_added'] = df['date_added'].dt.year
else:
    df['year_added'] = df['release_year']

print(f"Dataset Loaded: {len(df)} titles.")
df.head()

Dataset Loaded: 5850 titles.


,id,title,type,description,release_year,age_certification,runtime,listed_in,country,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,year_added
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,[documentation],[US],1.0,NaN,NaN,NaN,0.600,NaN,1945
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"[drama, crime]",[US],NaN,tt0075314,8.2,808582.0,40.965,8.179,1976
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"[drama, action, thriller, european]",[US],NaN,tt0068473,7.7,107673.0,10.010,7.300,1972
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"[fantasy, action, comedy]",[GB],NaN,tt0071853,8.2,534486.0,15.461,7.811,1975
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"[war, action]","[GB, US]",NaN,tt0061578,7.7,72662.0,20.398,7.600,1967


## 2. High-Level Metrics

Analyzing the balance between Movies and TV Shows.

In [11]:
content_counts = df['type'].value_counts().reset_index()
content_counts.columns = ['Type', 'Count']

fig_pie = px.pie(
    content_counts, 
    values='Count', 
    names='Type', 
    title="Library Composition: Movies vs TV Shows",
    color_discrete_sequence=['#E50914', '#444444']
)
fig_pie.show()

## 3. Maturity Profile

Understanding the distribution of age certifications.

In [12]:
rating_counts = df['age_certification'].value_counts().reset_index()
rating_counts.columns = ['Rating', 'Count']

fig_rating = px.bar(
    rating_counts, 
    x='Rating', 
    y='Count', 
    title="Content Maturity Profile",
    color_discrete_sequence=['#E50914']
)
fig_rating.show()

## 4. Genre Composition

Deep dive into the genres using a Sunburst chart.

In [13]:
sun_df = df[['type', 'listed_in']].explode('listed_in')
top_type_genres = sun_df.groupby(['type', 'listed_in']).size().reset_index(name='count')
top_type_genres = top_type_genres.sort_values(['type', 'count'], ascending=[True, False]).groupby('type').head(8)

fig_sun = px.sunburst(
    top_type_genres, 
    path=['type', 'listed_in'], 
    values='count',
    color='type', 
    color_discrete_map={'MOVIE': '#E50914', 'SHOW': '#444444'},
    title="Genre Distribution by Content Type"
)
fig_sun.show()

## 5. Acquisition Trends

How fast is the Netflix library growing?

In [14]:
trend_df = df.groupby('year_added').size().reset_index(name='count')
trend_df = trend_df[trend_df['year_added'] > 2000] # Focus on modern era
trend_df.columns = ['Year', 'Titles Added']

fig_trend = px.area(
    trend_df, 
    x='Year', 
    y='Titles Added', 
    title="Library Growth Over Time",
    color_discrete_sequence=['#E50914']
)
fig_trend.show()

## 6. Global Production Hubs

Identifying the top countries contributing content.

In [15]:
country_counts = df['country'].explode().value_counts().head(10).reset_index()
country_counts.columns = ['Country', 'Count']

fig_country = px.bar(
    country_counts, 
    x='Count', 
    y='Country', 
    orientation='h', 
    title="Top 10 Global Production Hubs",
    color_discrete_sequence=['#E50914']
)
fig_country.update_layout(yaxis={'categoryorder':'total ascending'})
fig_country.show()

## 7. Talent Analytics

Leveraging `credits.csv` to identify the most frequent actors across the Netflix library.

In [16]:
credits_df = pd.read_csv('credits.csv')
cast_counts = credits_df[credits_df['role'] == 'ACTOR']['name'].value_counts().head(10).reset_index()
cast_counts.columns = ['Actor', 'Count']

fig_cast = px.bar(
    cast_counts, 
    x='Count', 
    y='Actor', 
    orientation='h', 
    title="Top 10 Actors Appearing in Netflix Titles",
    color_discrete_sequence=['#E50914']
)
fig_cast.update_layout(yaxis={'categoryorder':'total ascending'})
fig_cast.show()